# 10 docs lint

Question: does the docs lint of Golem, `.github/scripts/docs_lint.py`, pass the documentation of
this revision, and does it refuse every break of its seven rules with the path and the line of the
break, while it leaves alone what the rules leave out. And do the pages state what the tree holds,
where they name a fact of the tree that no rule of the lint reads.

CONTRIBUTING.md section 4 states the rules of the prose, and the docstring of the script numbers
them with the two maps: the index of the wiki links every page, and section 4 of
`Documentation/testing.md` names every entry of `tests/matrix.yaml`. The notebook runs the lint on
the tree, then builds a small tree that follows every rule and changes it once per case in a copy
of it. A case names the path, the line and the rule that the lint must report, or none when the
copy must pass. The last sections read the pages beside the files they describe, one section for
each fact.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

## Setup

The cell finds the repository, prints the version of Python and the path of the lint, and defines
the small tree: a wiki of an index and two pages, one of them in a folder,
`Documentation/testing.md` with its section 4, a matrix of three entries, one of them with an id of
its own, CONTRIBUTING.md and README.md. The tree holds every form that the rules leave out: front
matter, fenced code, code spans, HTML comments, links and URLs, list markers, a hyphen inside a
word, a long row of a table and a long line that a long link makes unbreakable. It also holds a
Markdown file outside the four places the lint reads, which breaks every rule.

In [ ]:
import re
import shutil
import sys
import tempfile
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
LINT = P.root / ".github" / "scripts" / "docs_lint.py"
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-10-"))

print("python:", sys.version.split()[0])
print("lint:", P.rel(LINT))

FRONT = ("---\n# SPDX-License-Identifier: AGPL-3.0-only\n# SPDX-FileCopyrightText: 2026 Florian Kleber\n"
         "type: Spec\ntitle: %s\n"
         "description: A page of the small tree, whose front matter may run past the width of a line, since the "
         "lint leaves the front matter out.\n---\n\n")
HEAD = "<!-- SPDX-License-Identifier: AGPL-3.0-only -->\n<!-- SPDX-FileCopyrightText: 2026 Florian Kleber -->\n"
LONG_LINK = "[a page with a long address](https://example.com/" + "a" * 70 + ")"

FILES = {
    "okf/index.md": FRONT % "Index" + (
        "# Index\n\n"
        "The index of the small wiki.\n\n"
        "## Page map\n\n"
        "| Page | What it holds |\n"
        "| --- | --- |\n"
        "| [Page](page.md) | The page with every form that the rules leave out |\n"
        "| [Deep](plugins/deep.md#a-section) | A page in a folder, linked to a section |\n"),
    "okf/page.md": FRONT % "Page" + (
        "# Page\n\n"
        "A rule of the wiki keeps a well-known form, and people are they and them.\n\n"
        "- A list item starts with its marker.\n"
        "1. So does a numbered one.\n\n"
        "A code span such as `a; b -- c! he` and a URL such as https://example.com/a;b!c stay out.\n"
        "<!-- A comment; with a semicolon - and a dash! -->\n\n"
        "```text\n"
        "A fenced line; with a semicolon - a dash -- and a mark! She wrote it, and it runs past the width of a line.\n"
        "```\n\n"
        "| A row of a table | may run past the width of a line, since a row of a table is one line in the source of the page |\n\n"
        "A line that holds %s may run past the width.\n" % LONG_LINK),
    "okf/plugins/deep.md": FRONT % "Deep" + "# Deep\n\n## A section\n\nA page in a folder of the wiki.\n",
    "Documentation/testing.md": HEAD + (
        "# Testing\n\n"
        "## 3. Writing a notebook\n\n"
        "A notebook gets an entry in the matrix and a row below.\n\n"
        "## 4. The notebooks\n\n"
        "| Notebook | Question |\n"
        "| --- | --- |\n"
        "| `01_first` | Does the first notebook pass? |\n"
        "| `02_second` | Does the second notebook pass? |\n"
        "| `03_third_quick` | Does the third notebook pass in its quick form? |\n"),
    "tests/matrix.yaml": (
        "# SPDX-License-Identifier: AGPL-3.0-only\n# SPDX-FileCopyrightText: 2026 Florian Kleber\n\n"
        "entries:\n"
        "  - notebook: 01_first\n"
        "    cell_timeout: 300\n\n"
        "  - notebook: tests/notebooks/02_second.ipynb\n\n"
        "  - notebook: 03_third\n"
        "    id: 03_third_quick\n"
        "    parameters:\n"
        "      quick: true\n"),
    "CONTRIBUTING.md": HEAD + "# Contributing\n\nThe rulebook of a change.\n",
    "README.md": HEAD + "# Fixture\n\nA small tree for the docs lint.\n",
    "tools/notes.md": "# Notes\n\nA file the lint does not read; it breaks -- every rule! He wrote it. " + "x " * 60 + "\n",
}


def write(tree, path, text):
    target = tree / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8")


def base_tree(name):
    # A tree that follows every rule, in a directory of its own.
    tree = SCRATCH / name
    for path, text in FILES.items():
        write(tree, path, text)
    return tree


def append(tree, path, text):
    # Appends a line to a file and answers its number.
    target = tree / path
    lines = target.read_text(encoding="utf-8").rstrip("\n").split("\n") + [text]
    target.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return len(lines)


def line_of(tree, path, text):
    # The number of the first line of a file that is exactly the text.
    return (tree / path).read_text(encoding="utf-8").split("\n").index(text) + 1


def judge(tree):
    # The exit status of the lint on the tree and its failures as (path, line, rule).
    result = run(["python3", "-I", LINT, "--root", tree])
    failures = [(m[1], int(m[2]), int(m[3])) for m in re.finditer(r"^docs_lint: (\S+):(\d+): rule (\d+): ", result.stdout, re.M)]
    return result, failures


tree = base_tree("base")
result, failures = judge(tree)
print(result.stdout.strip())
assert result.returncode == 0 and not failures, "the small tree breaks a rule:\n" + result.stdout + result.stderr
assert "files 6," in result.stdout, "the lint read other files than the six of the four places:\n" + result.stdout

## The tree

The documentation of this revision follows every rule: the lint reads the wiki, `Documentation/`,
CONTRIBUTING.md and README.md, reports nothing and exits 0.

In [ ]:
tree_result = run(["python3", "-I", LINT])
print(tree_result.stdout.strip())
assert tree_result.returncode == 0, "the docs lint fails the tree:\n" + tree_result.stdout + tree_result.stderr
assert re.search(r"^docs_lint: files \d+, failures 0$", tree_result.stdout, re.M), tree_result.stdout

## Every rule, broken once

Each case changes one file of a copy of the small tree. A break must be refused with exactly one
failure, at the path and the line of the break and with its rule, and the lint must exit 1. A page
that the index does not link is reported at the heading of the page map of the index, and an entry
of the matrix without its row at the line of the entry. A form that the rules leave out must pass
with exit 0.

In [ ]:
def wiki(text, path="okf/page.md"):
    return lambda tree: (path, append(tree, path, text))


def orphan(path):
    def change(tree):
        write(tree, path, FRONT % "Orphan" + "# Orphan\n\nA page that the index does not link.\n")
        return "okf/index.md", line_of(tree, "okf/index.md", "## Page map")
    return change


def entry(lines):
    def change(tree):
        number = append(tree, "tests/matrix.yaml", lines[0])
        for text in lines[1:]:
            append(tree, "tests/matrix.yaml", text)
        return "tests/matrix.yaml", number
    return change


def row_in_section_3(tree):
    target = tree / "Documentation" / "testing.md"
    text = target.read_text(encoding="utf-8").replace(
        "A notebook gets an entry in the matrix and a row below.\n",
        "A notebook gets an entry in the matrix and a row below.\n\n| `04_fourth` | Not in section 4 |\n")
    target.write_text(text, encoding="utf-8")
    return entry(["  - notebook: 04_fourth"])(tree)


def passes(change):
    return lambda tree: (change(tree), None)[1]


CASES = [
    # name, change, which answers the path and the line of the break, and the rule, or None to pass
    ("the small tree", lambda tree: None, None),
    ("1: a semicolon in a page of the wiki", wiki("One clause; and another."), 1),
    ("1: a semicolon in a page of Documentation", wiki("One clause; and another.", "Documentation/testing.md"), 1),
    ("2: an em dash in CONTRIBUTING.md", wiki("A rule — and its reason.", "CONTRIBUTING.md"), 2),
    ("2: an en dash in README.md", wiki("A rule – and its reason.", "README.md"), 2),
    ("2: a hyphen with a space on each side", wiki("A rule - and its reason."), 2),
    ("2: a double hyphen between two words", wiki("A rule--and its reason."), 2),
    ("3: an exclamation mark", wiki("The page works!"), 3),
    ("4: he for a person", wiki("The owner said he would look."), 4),
    ("4: her for a person", wiki("The owner read her page."), 4),
    ("5: a line of 101 columns", wiki("A line of prose" + " word" * 17 + "s"), 5),
    ("5: a list item of 101 columns", wiki("- A list item" + " word" * 17 + " ab"), 5),
    ("6: a page of the wiki that the index does not link", orphan("okf/orphan.md"), 6),
    ("6: a page in a folder that the index does not link", orphan("okf/plugins/orphan.md"), 6),
    ("7: an entry of the matrix without a row", entry(["  - notebook: 04_fourth", "    cell_timeout: 300"]), 7),
    ("7: an entry whose own id has no row", entry(["  - notebook: 01_first", "    id: 01_first_quick"]), 7),
    ("7: a row of the entry outside section 4", row_in_section_3, 7),
    ("a line of 100 columns", passes(wiki("A line of prose" + " word" * 17)), None),
    ("a hyphen inside a word and a list marker", passes(wiki("- A well-known rule of a list.")), None),
    ("a semicolon in a code span, a URL and a comment", passes(wiki("A span `a; b`, https://example.com/a;b and <!-- c; d -->.")), None),
    ("a fenced block that breaks every rule", passes(wiki("```\nOne; two -- three! He wrote it.\n```")), None),
    ("a long row of a table", passes(wiki("| A row | " + "x " * 60 + "|")), None),
    ("a long line with a link longer than 60 columns", passes(wiki("A line with %s and more prose to run long." % LONG_LINK)), None),
    ("a Markdown file outside the four places", passes(lambda tree: write(tree, "tests/notes.md", "One; two -- three!\n")), None),
    ("an entry with a row in section 4", passes(entry(["  - notebook: 01_first"])), None),
]
print("cases:", len(CASES))

In [ ]:
rows, wrong = [], []
for index, (name, change, rule) in enumerate(CASES):
    tree = base_tree("case-%02d" % index)
    where = change(tree)
    result, failures = judge(tree)
    if rule is None:
        expected = []
        ok = result.returncode == 0 and not failures
    else:
        expected = [(where[0], where[1], rule)]
        ok = result.returncode == 1 and failures == expected
    rows.append((name, ", ".join("%s:%d rule %d" % failure for failure in expected) or "pass",
                 "ok" if ok else "WRONG"))
    if not ok:
        wrong.append((name, expected, result.returncode, result.stdout + result.stderr))

width = max(len(row[0]) for row in rows)
print("%-*s  %-40s %s" % (width, "case", "expected", ""))
for row in rows:
    print("%-*s  %-40s %s" % (width, *row))
for name, expected, code, output in wrong:
    print("\n--- %s, expected %s, exit %d\n%s" % (name, expected or "a pass", code, output))
assert not wrong, "%d of %d cases went wrong" % (len(wrong), len(CASES))

## The license of the ports

The README and the plan state the license of Golem. The ports of `ports.toml` keep the licenses of
their upstreams and state them in their own `Cargo.toml`, so the License section of the README must
name `ports.toml` and the license of every port, and neither page may say that every `Cargo.toml`
states the AGPL 3 only.

In [ ]:
import tomllib


def page(path):
    return (P.root / path).read_text(encoding="utf-8")


def section(path, heading):
    # The text of a section of a Markdown page, up to the next heading of its level or above.
    level = len(heading.split(" ")[0])
    match = re.search(r"^%s\n(.*?)(?=^#{1,%d} |\Z)" % (re.escape(heading), level), page(path), re.M | re.S)
    assert match, "%s has no section %r" % (path, heading)
    return match.group(1)


PORTS = tomllib.loads(page("ports.toml"))["port"]
licenses = {port["name"]: tomllib.loads(page(port["path"] + "/Cargo.toml"))["package"].get("license") for port in PORTS}
print("ports:", licenses)
assert all(isinstance(value, str) for value in licenses.values()), "a port inherits the license of the workspace: %s" % licenses

readme = section("README.md", "## License")
assert "[ports.toml](ports.toml)" in readme, "the License section of README.md does not name ports.toml"
for license in sorted(set(licenses.values())):
    assert "`%s`" % license in readme, "the License section of README.md does not name %s, the license of a port" % license
for path in ("README.md", "okf/plan.md"):
    claim = re.search(r"every `Cargo\.toml`[^.]*`AGPL-3\.0-only`", page(path))
    assert not claim, "%s says that every Cargo.toml states AGPL-3.0-only: %r" % (path, claim.group(0))
print("the README names ports.toml and", ", ".join(sorted(set(licenses.values()))))

## Where the SPDX tags stand

`Documentation/licensing.md` section 3 places the tags of a file: on its first lines, one line lower
under a `#!` line or the `---` of a front matter, and in a `.license` file beside a file that cannot
hold a comment, while `REUSE.toml` gives `NOTICE` alone its tags. Section 4 keeps in `LICENSES/` the
text of every license that a file names, and section 6 lets a port translate its upstream. The cell
reads the files of the tree and refuses a sentence of the README or the plan that states a simpler
rule, which the tree breaks.

In [ ]:
import os


def tree_files():
    # The files of the tree relative to its root, from git, or else from a walk that leaves out
    # the folders that .gitignore names.
    listing = run(["git", "-C", P.root, "ls-files", "-z"])
    if listing.returncode == 0:
        return sorted(name for name in listing.stdout.split("\0") if name and (P.root / name).is_file())
    skip = {".git", "out", ".venv", "__pycache__", ".ipynb_checkpoints", ".cache", "target", ".gradle", "build"}
    found = []
    for directory, folders, names in os.walk(P.root):
        folders[:] = [folder for folder in folders if folder not in skip]
        found += [os.path.relpath(os.path.join(directory, name), P.root) for name in names]
    return sorted(found)


def text_of(name):
    try:
        return (P.root / name).read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return None


TREE = tree_files()
TEXTS = {name: text for name in TREE if (text := text_of(name)) is not None}


def tag_line(text):
    # The line of the SPDX license tag among the first three, or None.
    for number, line in enumerate(text.split("\n")[:3], 1):
        if "SPDX-License-Identifier:" in line:
            return number
    return None


lower = sorted(name for name, text in TEXTS.items() if tag_line(text) == 2)
beside = sorted(name[: -len(".license")] for name in TREE if name.endswith(".license"))
annotated = []
for entry in tomllib.loads(page("REUSE.toml")).get("annotations", []):
    annotated += [entry["path"]] if isinstance(entry["path"], str) else entry["path"]
named = set()
for text in TEXTS.values():
    for expression in re.findall(r"SPDX-License-Identifier: ([A-Za-z0-9.+() -]+)", text):
        named.update(set(re.findall(r"[A-Za-z0-9.+-]+", expression)) - {"AND", "OR", "WITH"})
texts = sorted(path.stem for path in (P.root / "LICENSES").glob("*.txt"))
print("files %d, with the tag one line lower %d, with a .license file %d, annotated in REUSE.toml %s"
      % (len(TREE), len(lower), len(beside), annotated))

readme, plan = section("README.md", "## License"), page("okf/plan.md")
if lower or beside:
    for path, text in (("README.md", readme), ("okf/plan.md", plan)):
        claim = re.search(r"on its first line|first line of every file", text)
        assert not claim, ("%s says that every file names its license on its first line, but %d files carry the tag one "
                           "line lower and %d in a .license file" % (path, len(lower), len(beside)))
    assert "`.license`" in readme, "the License section of README.md does not name the .license file"
if beside and not set(beside) & set(annotated):
    claim = re.search(r"cannot hold a comment[^.]*`REUSE\.toml`", plan)
    assert not claim, "okf/plan.md gives REUSE.toml the files that cannot hold a comment: %r" % claim.group(0)
    assert "`.license`" in plan, "okf/plan.md does not name the .license file"
assert set(texts) <= named, "LICENSES/ holds the texts %s, which no file names" % sorted(set(texts) - named)
claim = re.search(r"license a dependency", plan)
assert not claim, "okf/plan.md says that LICENSES/ holds the texts that dependencies bring: %r" % claim.group(0)
if any(port["upstream"].startswith("EssentialsX") for port in PORTS):
    claim = re.search(r"not copy or translate", plan)
    assert not claim, "okf/plan.md says that Golem does not translate EssentialsX, whose ports keep its license"
print("the README and the plan place the tags as licensing.md does")

## What the commit lint checks of a sign off

`okf/decisions.md` records the ruling that every commit is signed off in the name of the holder,
and its status says what checks it. The cell hands the commit lint a message that keeps every rule
and whose sign off names someone else. The status may say that the commit lint checks the name
exactly when the lint refuses that message under its rule 6.

In [ ]:
COMMIT_LINT = P.root / ".github" / "scripts" / "commit_lint.py"
rows = [line for line in page("okf/decisions.md").split("\n") if "signed off in the name of the holder" in line]
assert len(rows) == 1, "okf/decisions.md holds %d rows of the ruling on the sign off" % len(rows)
status = rows[0].split(" | ")[2]
print("status:", status)

message = SCRATCH / "foreign-sign-off.txt"
message.write_text("docs(okf): sign a message off in the name of someone else\n\n"
                   "The message follows every rule of the commit lint, and its sign off\n"
                   "names a person who is not the holder of Golem.\n\n"
                   "Signed-off-by: Someone Else <someone@example.org>\n", encoding="utf-8")
result = run(["python3", "-I", COMMIT_LINT, "--root", P.root, "--message-file", message])
print("the commit lint exits %d on a sign off of someone else" % result.returncode, result.stdout.strip())
refuses = result.returncode != 0
assert not refuses or "rule 6" in result.stdout, "the commit lint refuses the message for another rule:\n" + result.stdout + result.stderr
claims = "not whose name" not in status
assert claims == refuses, ("okf/decisions.md says that the commit lint %s the name of the sign off, but the lint %s a sign off of "
                           "someone else" % ("checks" if claims else "does not check", "refuses" if refuses else "passes"))

## Who checks the permissions of a command

A plugin can ask `golem-permissions` only when the program `golem` links the crate. The cell
follows the dependencies of the crate `golem` through the crates of the workspace. While none of
them names `golem-permissions`, every sentence of the pages of the bridge, the plugins and the
permissions that has a plugin check, judge, ask or grant through the crate must say that this is
not so yet, with one of the words once, yet, until or will.

In [ ]:
def links(name, seen=None):
    # The crates of the workspace that a crate links, itself included.
    seen = set() if seen is None else seen
    seen.add(name)
    for dependency in tomllib.loads(page("crates/%s/Cargo.toml" % name)).get("dependencies", {}):
        if dependency not in seen and (P.root / "crates" / dependency / "Cargo.toml").is_file():
            links(dependency, seen)
    return seen


def sentences(path):
    # The sentences of the prose of a page, without its front matter, fenced code and tables.
    text = page(path)
    if text.startswith("---\n"):
        text = text.split("\n---\n", 1)[1]
    text = re.sub(r"^```.*?^```", "", text, flags=re.M | re.S)
    prose = " ".join(line.strip() for line in text.split("\n") if not line.startswith("|"))
    return re.split(r"(?<=[.?])\s+(?=[A-Z`\[])", prose)


program = links("golem")
print("the program golem links:", ", ".join(sorted(program)))
if "golem-permissions" in program:
    print("golem links golem-permissions, so the pages may say that a plugin checks through it")
else:
    false = []
    for path in ("okf/bridge.md", "okf/plugins/index.md", "okf/plugins/permissions.md"):
        for sentence in sentences(path):
            if ("golem-permissions" in sentence and re.search(r"\bplugins?\b", sentence)
                    and re.search(r"\b(checks?|judges?|asks?|grants?)\b", sentence)):
                said = bool(re.search(r"\b(once|yet|until|will)\b", sentence, re.I))
                print("%-26s %s  %s" % (path, "ok   " if said else "FALSE", sentence[:150]))
                if not said:
                    false.append((path, sentence))
    assert not false, "a page says that a plugin checks through golem-permissions, which golem does not link:\n" + "\n".join(
        "%s: %s" % item for item in false)

## The addresses in the tree

CONTRIBUTING.md section 9 lets no mail address of a third party into the repository and names the
places where the addresses of others stand: the address of the upstream of a port in NOTICE,
`ports.toml`, the SPDX lines of the files of the port and the cases of the license lint, the
address of an agent in a `Co-Authored-By` line of the cases of the commit lint, and an address of a
domain reserved for examples. The cell reads every file of the tree and holds every address it
finds, beside the one of the holder, to those places.

In [ ]:
ADDRESS = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+")
LICENSE_CASES = "tests/notebooks/01_license_lint.ipynb"
COMMIT_CASES = "tests/notebooks/02_commit_lint.ipynb"


def reserved(domain):
    # The domains of RFC 2606 and RFC 6761 for examples and tests, which name nobody.
    return domain in ("example.org", "example.com", "example.net") or domain.rsplit(".", 1)[-1] in (
        "example", "invalid", "test", "localhost")


rule = section("CONTRIBUTING.md", "## 9. Nothing private")
for word in ("NOTICE", "`ports.toml`", "`Co-Authored-By`", "`example.org`"):
    assert word in rule, "section 9 of CONTRIBUTING.md does not name %s" % word
holder = re.search(r"<([^>]+)>", tomllib.loads(page("REUSE.toml"))["annotations"][0]["SPDX-FileCopyrightText"]).group(1)
upstream = {found.group(1): port["path"] for port in PORTS for name in port["holders"] if (found := re.search(r"<([^>]+)>", name))}
print("the holder:", holder, " the upstreams:", upstream)

counts, wrong = {}, []
for name, text in TEXTS.items():
    for number, line in enumerate(text.split("\n"), 1):
        for address in ADDRESS.findall(line):
            if address == holder:
                kind = "holder"
            elif reserved(address.split("@", 1)[1].lower()):
                kind = "example"
            elif address in upstream and (name in ("NOTICE", "ports.toml", LICENSE_CASES) or (
                    name.startswith(upstream[address] + "/") and "SPDX-FileCopyrightText:" in line)):
                kind = "upstream"
            elif name == COMMIT_CASES and re.search(r"Co-Authored-By: [^<]*<%s>" % re.escape(address), line):
                kind = "agent"
            else:
                wrong.append("%s:%d: %s" % (name, number, address))
                continue
            counts[kind] = counts.get(kind, 0) + 1
print("addresses by kind:", counts)
assert not wrong, "addresses that section 9 of CONTRIBUTING.md does not allow:\n" + "\n".join(wrong)

## Verdict

The cell clears the scratch trees and prints the count. A run that reaches it passed every case
and found every page in agreement with the tree.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the docs lint passes the tree and judges %d cases as expected, and the pages state what the tree holds" % len(CASES))